In [3]:
############################################################################################
############ Clean and Processing script for the ERA5-Land data from GEE ###################
############################################################################################

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pytz # for converting era5 utc time to est/edt time
from datetime import datetime, timedelta # for getting hourly mean radiation values from accumulations

# define working dirs
#parent_folder = os.path.dirname(os.path.dirname(__file__))
#parent_folder = os.path.dirname(os.getcwd())
parent_folder = r"D:\DDL\Durham_ML\Other_Cities\ML_inputs"
print(parent_folder)

#Timezones - Baltimore,  Durham = EDT , ***************** CHECK TIMEZONE OF CITY YOU ARE LOOKING AT! ***************************
# daylight time for summer in places with daylight saving time
edt = pytz.timezone('US/Eastern')
cdt = pytz.timezone('America/Chicago')
mst = pytz.timezone('America/Phoenix')

## define function to get rh from era5 dew point and air temperatures
def calculate_relative_humidity(t2m, d2m):
    """
    calculate relative humidity from 2m air and dew point temps
    August-Roche-Magnus equation (https://doi.org/10.5194/tc-2023-8)
        t2m: 2m air temp in degC
        d2m: 2m dew point temp in degC
        
        return: relative humidity in %
    """
    # Calculate saturation vapor pressure (e_s) for air temp
    if t2m > 0:
        e_s = 6.11 * np.exp((17.62 * t2m) / (t2m + 243.12))
    else:
        e_s = 6.11 * np.exp((22.46 * t2m) / (t2m + 272.62))
    
    # Calculate actual vapor pressure (e_d) for dew point temp
    if d2m > 0:
        e_d = 6.11 * np.exp((17.62 * d2m) / (d2m + 243.12))
    else:
        e_d = 6.11 * np.exp((22.46 * d2m) / (d2m + 272.62))
    
    # Calculate relative humidity
    rh = 100 * (e_d / e_s)
    
    return rh


# read in sensor location file
for city_name in ['Baltimore']:
    era5_clean_path =os.path.join(parent_folder,f"ERA5/ERA5_land_{city_name}_mean_clean.csv")
    era5_mean_raw = pd.read_csv(os.path.join(parent_folder,f"gee_downloaded/{city_name}_ERA5_mean.csv"))

    era5_clean =era5_mean_raw.copy()
    #era5_clean.drop(columns = ['.geo'], inplace = True)
    # Split the 'system:index' column to extract the date and custom hour
    era5_clean['date'] = pd.to_datetime(era5_clean['system:index'].str.split('T').str[0], format='%Y%m%d')
    era5_clean['date'] = era5_clean['date'].dt.date

    # Extract the hour part right after 'T' and format it as HH:00
    era5_clean['hour'] = era5_clean['system:index'].str.extract(r'T(\d{2})')[0] + ":00"

    # convert the era5 time to EDT/EST (currently it is UTC)
    # create a datetime using the date and hour, so the UTC to EST conversion can be carried out
    era5_clean['utc_datetime'] = pd.to_datetime(era5_clean['date'].astype(str)+ ' ' + era5_clean['hour'])

    # Localize the datetime to UTC , ***************** CHECK TIMEZONE OF CITY YOU ARE LOOKING AT! ***************************
    era5_clean['utc_datetime'] = era5_clean['utc_datetime'].dt.tz_localize('UTC')
    era5_clean['edt_datetime'] = era5_clean['utc_datetime'].dt.tz_convert(edt)
    era5_clean['edt_datetime'] = era5_clean['edt_datetime'].dt.tz_localize(None)    

    # create the hour column in the local time zone 
    era5_clean['local_datetime'] = era5_clean['edt_datetime'] 
    
    era5_clean['Custom_Hour'] = era5_clean['local_datetime'].dt.strftime('%H:%M')    
    era5_clean['local_date'] = era5_clean['local_datetime'].dt.date


    # convert the temperatures to celcius from K
    era5_clean['temperature_2m'] = era5_clean['temperature_2m'] -273.15
    era5_clean['dewpoint_temperature_2m'] = era5_clean['dewpoint_temperature_2m'] -273.15    

    era5_clean['relative_humidity_2m'] = era5_clean.apply(lambda row: calculate_relative_humidity(
                                        row['temperature_2m'], row['dewpoint_temperature_2m']), axis=1)    

    # add in the windspeed based on the two vectors
    era5_clean['windspeed_10m'] = np.sqrt(era5_clean['u_component_of_wind_10m']**2 + era5_clean['v_component_of_wind_10m']**2)

    ## convert radiation into watts
    era5_clean['surface_net_solar_radiation_hourly'] = era5_clean.surface_net_solar_radiation_hourly / 3600
    era5_clean['surface_net_thermal_radiation_hourly'] = era5_clean.surface_net_thermal_radiation_hourly / 3600
    era5_clean['surface_solar_radiation_downwards_hourly'] = era5_clean.surface_solar_radiation_downwards_hourly / 3600
    era5_clean['surface_thermal_radiation_downwards_hourly'] = era5_clean.surface_thermal_radiation_downwards_hourly / 3600

    ## add in accumulation of daily incoming solar radiation 
    ## want to add the accumulation based on the local datetime 
    for i in range(len(era5_clean)):
        if era5_clean.iloc[i]['local_datetime'].to_pydatetime().hour == 1:
            era5_clean.loc[i, 'surface_solar_radiation_downwards_accumulation'] = (era5_clean.iloc[i].
                                                                   surface_solar_radiation_downwards_hourly)
    
        # the data for the 31st may does not have a full day so set this to nan as cannot calculate the accumulation
        elif era5_clean.iloc[i]['local_datetime'].strftime('%Y-%m-%d') == '2024-05-31':
            era5_clean.loc[i, 'surface_solar_radiation_downwards_accumulation'] = np.nan
    
        else:
        
            hour_to_accumulate_on = era5_clean.iloc[i]['local_datetime'].to_pydatetime()
            hour_before = era5_clean.iloc[i]['local_datetime'].to_pydatetime() - timedelta(hours=1)
            date = hour_before.date() # date of hour before, incase the measurement is midnight
        
            accumulation_dw_solar = era5_clean[(era5_clean.local_datetime <= hour_to_accumulate_on)&(
                              era5_clean.local_datetime.dt.date == date)&(
                               era5_clean.Custom_Hour !='00:00')].surface_solar_radiation_downwards_hourly.sum()

        
            era5_clean.loc[i, 'surface_solar_radiation_downwards_accumulation'] = accumulation_dw_solar


    # Drop the original 'system:index' column if no longer needed
    #era5_clean = era5_clean.drop(columns=['system:index'])
    era5_clean.to_csv(era5_clean_path,index=False)

D:\DDL\Durham_ML\Other_Cities\ML_inputs
